In [1]:
import os, csv, math, warnings, pickle, glob, re, shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import pyeeg as pe
from tqdm import tqdm

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score, classification_report, confusion_matrix

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# ─────────────────────────────────────────────────────────────────────────────
# 0. SETUP
# ─────────────────────────────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {device}")

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
warnings.filterwarnings("ignore")

BASE_DIR = r"C:\Users\PC\Desktop\Facultate\LICENTA\notebooks\DeepGAT"

FOLDERS = {
    "calm":     os.path.join(BASE_DIR, "recordings_calm_cleaned"),
    "happy":    os.path.join(BASE_DIR, "recordings_happy_cleaned"),
    "sad":      os.path.join(BASE_DIR, "recordings_sad_cleaned"),
    "stressed": os.path.join(BASE_DIR, "recordings_stressed_cleaned"),
}

# NEW cache folder name — guarantees stale STEP=16 files are never loaded
OUT_FEAT = os.path.join(BASE_DIR, "data_extracted_v2")
OUT_WORK = os.path.join(BASE_DIR, "gat_output_robust")
os.makedirs(OUT_FEAT, exist_ok=True)
os.makedirs(OUT_WORK, exist_ok=True)

# ─────────────────────────────────────────────────────────────────────────────
# 1. CONFIG & MAPPING
# ─────────────────────────────────────────────────────────────────────────────
LABEL_MAP = {
    "calm":     [0, 1],  # Low Arousal,  High Valence
    "happy":    [1, 1],  # High Arousal, High Valence
    "sad":      [0, 0],  # Low Arousal,  Low Valence
    "stressed": [1, 0],  # High Arousal, Low Valence
}

CHANNEL_NAMES = [
    "Fp1", "Fp2", "C3", "C4", "P7", "P8", "O1", "O2",
    "F7",  "F8",  "F3", "F4", "T7", "T8", "P3", "P4",
]

BAND_EDGES  = [4, 8, 12, 16, 25, 45]
BAND_LABELS = ["Theta", "Alpha", "BetaL", "BetaH", "Gamma"]

FS                 = 125
WINDOW             = 250    # 2-second windows at 125 Hz
STEP               = 125    # non-overlapping (0% overlap)
ARTIFACT_THRESHOLD = 100.0  # µV — windows exceeding this are discarded

N_CH    = len(CHANNEL_NAMES)   # 16
N_BANDS = len(BAND_EDGES) - 1  # 5
N_FEATS = N_BANDS * 2          # 10 (5 BP + 5 DE)
FEAT_LABELS = (
    [f"BP_{b}" for b in BAND_LABELS] +
    [f"DE_{b}" for b in BAND_LABELS]
)

# ─────────────────────────────────────────────────────────────────────────────
# 2. FEATURE EXTRACTION  (raw — no per-file normalization)
# ─────────────────────────────────────────────────────────────────────────────
def differential_entropy(sig, band_edges, fs):
    de, fft_full = [], np.fft.rfft(sig)
    freqs = np.fft.rfftfreq(len(sig), 1.0 / fs)
    for lo, hi in zip(band_edges[:-1], band_edges[1:]):
        mask = (freqs >= lo) & (freqs < hi)
        f = np.zeros_like(fft_full)
        f[mask] = fft_full[mask]
        var = np.var(np.fft.irfft(f, n=len(sig))) + 1e-10
        de.append(0.5 * np.log(2 * np.pi * np.e * var))
    return de


def extract_features_from_csv(csv_path, category):
    filename = os.path.basename(csv_path)
    out_name = f"{category}_{filename.replace('.csv', '.npy')}"
    out_path = os.path.join(OUT_FEAT, out_name)

    if os.path.exists(out_path):
        return out_path

    df       = pd.read_csv(csv_path)
    eeg_cols = [col for col in df.columns if col in CHANNEL_NAMES]
    data     = df[eeg_cols].values.T        # (16, N_samples)

    label        = LABEL_MAP[category]
    raw_features = []
    n_rejected   = 0

    start = 0
    while start + WINDOW <= data.shape[1]:
        win_data = data[:, start: start + WINDOW]
        if np.abs(win_data).max() > ARTIFACT_THRESHOLD:
            n_rejected += 1
            start += STEP
            continue
        win_feats = []
        for ch in range(N_CH):
            sig   = win_data[ch]
            ch_f  = (
                list(pe.bin_power(sig, BAND_EDGES, FS)[0]) +
                differential_entropy(sig, BAND_EDGES, FS)
            )
            win_feats.append(ch_f)
        raw_features.append(win_feats)
        start += STEP

    if not raw_features:
        print(f"  ⚠ No valid windows in {filename} — skipped.")
        return None

    raw_features = np.array(raw_features, dtype=np.float32)  # (W, 16, 10)
    meta = [
        [raw_features[i], np.array(label, dtype=np.float32)]
        for i in range(len(raw_features))
    ]
    np.save(out_path, np.array(meta, dtype=object), allow_pickle=True)
    return out_path


print("Extracting features into data_extracted_v2 (fresh cache)...")
dataset_files = []
for cat, folder in FOLDERS.items():
    csv_files = glob.glob(os.path.join(folder, "*.csv"))
    for csv_file in tqdm(csv_files, desc=f"  {cat}"):
        npy_path = extract_features_from_csv(csv_file, cat)
        if npy_path is not None:
            dataset_files.append((npy_path, cat))

# ─────────────────────────────────────────────────────────────────────────────
# 3. LOAD WINDOWS + SANITY CHECK
# ─────────────────────────────────────────────────────────────────────────────
print("\nLoading windows...")
all_X, all_Y, all_groups = [], [], []

for fp, cat in dataset_files:
    arr      = np.load(fp, allow_pickle=True)
    filename = os.path.basename(fp)
    for row in arr:
        all_X.append(row[0])
        all_Y.append(row[1])
        all_groups.append(filename)

all_X      = np.array(all_X,      dtype=np.float32)
all_Y      = np.array(all_Y,      dtype=np.float32)
all_groups = np.array(all_groups)

n_files       = len(np.unique(all_groups))
n_windows     = len(all_X)
wins_per_file = n_windows / n_files
expected_wpf  = (7500 - WINDOW) // STEP

print(f"\n{'─'*55}")
print(f"  Unique files       : {n_files}")
print(f"  Total windows      : {n_windows:,}")
print(f"  Windows/file       : {wins_per_file:.1f}  (expected ~{expected_wpf})")
if wins_per_file > expected_wpf * 3:
    print(f"  !! WARNING: windows/file is {wins_per_file:.0f} but should be ~{expected_wpf}.")
    print(f"  !! The old STEP=16 cache is probably loaded. Check OUT_FEAT path.")
else:
    print(f"  STEP check         : OK")
print(f"{'─'*55}\n")

# ─────────────────────────────────────────────────────────────────────────────
# 4. MODEL DEFINITIONS
# ─────────────────────────────────────────────────────────────────────────────
class EEGDataset(Dataset):
    """
    Gaussian noise augmentation on training set only (noise_std=0 for val/test).
    Forces the model to learn robust patterns instead of memorising exact values.
    """
    def __init__(self, X, Y, noise_std=0.0):
        self.X         = torch.from_numpy(X).float()
        self.Y         = torch.from_numpy(Y).float()
        self.noise_std = noise_std

    def __len__(self):
        return len(self.Y)

    def __getitem__(self, idx):
        x = self.X[idx]
        if self.noise_std > 0:
            x = x + torch.randn_like(x) * self.noise_std
        return x, self.Y[idx]


class GATLayer(nn.Module):
    def __init__(self, in_features, out_features, num_heads=4,
                 attn_dropout=0.1, residual=True):
        super().__init__()
        self.H        = num_heads
        self.d        = out_features
        self.residual = residual

        self.W         = nn.Linear(in_features, num_heads * out_features, bias=False)
        self.a         = nn.Parameter(torch.empty(num_heads, 2 * out_features))
        nn.init.xavier_uniform_(self.a.unsqueeze(0))

        self.leaky     = nn.LeakyReLU(0.2)
        self.attn_drop = nn.Dropout(attn_dropout)
        self.bn        = nn.BatchNorm1d(out_features)

        if residual:
            self.res_proj = (
                nn.Linear(in_features, out_features, bias=False)
                if in_features != out_features else nn.Identity()
            )

    def forward(self, x):
        B, N, _ = x.shape
        h    = self.W(x).view(B, N, self.H, self.d)
        hi   = h.unsqueeze(2)
        hj   = h.unsqueeze(1)
        pair = torch.cat([
            hi.expand(B, N, N, self.H, self.d),
            hj.expand(B, N, N, self.H, self.d),
        ], dim=-1)
        e     = self.leaky(
            (pair * self.a.unsqueeze(0).unsqueeze(0).unsqueeze(0)).sum(-1)
        )
        alpha = self.attn_drop(F.softmax(e, dim=2))
        out   = torch.einsum("bqkh, bkhd -> bqhd", alpha, h).mean(dim=2)
        out   = self.bn(out.reshape(B * N, self.d)).reshape(B, N, self.d)
        out   = F.elu(out)
        if self.residual:
            out = out + self.res_proj(x)
        return out, alpha.permute(0, 3, 1, 2)


class TaskHead(nn.Module):
    def __init__(self, in_dim, dense, head_dropout=0.6):
        super().__init__()
        self.head = nn.Sequential(
            nn.Linear(in_dim, dense),
            nn.LayerNorm(dense),
            nn.GELU(),
            nn.Dropout(head_dropout),
            nn.Linear(dense, dense // 2),
            nn.GELU(),
            nn.Dropout(head_dropout * 0.5),
            nn.Linear(dense // 2, 1),
        )

    def forward(self, x):
        return self.head(x.mean(dim=1))


class DeepGAT(nn.Module):
    def __init__(self, n_channels, in_feats, backbone_dims, dense_size,
                 num_heads=4, attn_dropout=0.1, head_dropout=0.6):
        super().__init__()
        d0 = backbone_dims[0]
        self.input_proj = nn.Linear(in_feats, d0)
        self.ch_embed   = nn.Parameter(torch.randn(1, n_channels, d0) * 0.02)

        dims = [d0] + backbone_dims
        self.backbone = nn.ModuleList([
            GATLayer(dims[i], dims[i + 1], num_heads, attn_dropout, residual=True)
            for i in range(len(backbone_dims))
        ])
        self.head_aro = TaskHead(backbone_dims[-1], dense_size, head_dropout)
        self.head_val = TaskHead(backbone_dims[-1], dense_size, head_dropout)

    def forward(self, x, return_attn=False):
        x = F.gelu(self.input_proj(x))
        x = x + self.ch_embed
        backbone_attns = []
        for layer in self.backbone:
            x, aw = layer(x)
            backbone_attns.append(aw)
        out = torch.cat([self.head_aro(x), self.head_val(x)], dim=1)
        if return_attn:
            return out, backbone_attns
        return out


class LabelSmoothingBCE(nn.Module):
    """
    Label smoothing prevents the model from pushing logits to ±∞ to achieve
    near-zero loss on training data. Targets become 0.9/0.1 instead of 1.0/0.0,
    so there is always a small residual training loss.
    """
    def __init__(self, pw_aro, pw_val, smoothing=0.1):
        super().__init__()
        self.smoothing = smoothing
        self.loss_aro  = nn.BCEWithLogitsLoss(pos_weight=pw_aro)
        self.loss_val  = nn.BCEWithLogitsLoss(pos_weight=pw_val)

    def forward(self, logits, targets):
        t = targets * (1.0 - self.smoothing) + 0.5 * self.smoothing
        return (
            self.loss_aro(logits[:, 0], t[:, 0]) +
            self.loss_val(logits[:, 1], t[:, 1])
        ) / 2

# ─────────────────────────────────────────────────────────────────────────────
# 5. HYPERPARAMETERS
# ─────────────────────────────────────────────────────────────────────────────
BACKBONE_DIMS   = [32, 32]  # small — right-sized for 100 trials
DENSE_SIZE      = 64
NUM_HEADS       = 4
ATTN_DROPOUT    = 0.1
HEAD_DROPOUT    = 0.6       # was 0.5 in previous version

NOISE_STD       = 0.15      # σ of Gaussian noise injected into training features
LABEL_SMOOTHING = 0.1       # smooths hard 0/1 targets → 0.1/0.9

BATCH_SIZE      = 64        # small batch = more stochasticity = regularization
EPOCHS          = 80
LR              = 2e-4
WARMUP_EPOCHS   = 5
PATIENCE        = 15        # early stopping patience (val loss)

N_SPLITS        = 10        # 10-fold with 100 files → ~10 test files per fold

# ─────────────────────────────────────────────────────────────────────────────
# 6. K-FOLD TRAINING LOOP
# ─────────────────────────────────────────────────────────────────────────────
gkf = GroupKFold(n_splits=N_SPLITS)

fold_results_macro    = []
fold_train_loss_best  = []
fold_val_loss_best    = []
best_overall_f1       = 0.0
best_overall_state    = None

print(f"[!] Starting {N_SPLITS}-Fold GroupKFold (file-level splits)")
print(f"[!] noise_std={NOISE_STD} | label_smoothing={LABEL_SMOOTHING} | "
      f"patience={PATIENCE} | batch={BATCH_SIZE}\n")

for fold, (train_idx, test_idx) in enumerate(
        gkf.split(all_X, all_Y, groups=all_groups), 1):

    print(f"{'='*55}")
    print(f" Fold {fold}/{N_SPLITS} | "
          f"train={len(train_idx):,} | test={len(test_idx):,}")

    x_tr_raw, y_tr = all_X[train_idx], all_Y[train_idx]
    x_te_raw, y_te = all_X[test_idx],  all_Y[test_idx]

    # Global normalization — fit on train only, transform both
    scaler = StandardScaler()
    x_tr   = scaler.fit_transform(
        x_tr_raw.reshape(-1, N_FEATS)
    ).reshape(-1, N_CH, N_FEATS)
    x_te   = scaler.transform(
        x_te_raw.reshape(-1, N_FEATS)
    ).reshape(-1, N_CH, N_FEATS)

    with open(os.path.join(OUT_WORK, f"scaler_fold{fold}.pkl"), "wb") as f:
        pickle.dump(scaler, f)

    pw_aro = torch.tensor(
        [(y_tr[:, 0] == 0).sum() / max((y_tr[:, 0] == 1).sum(), 1)],
        dtype=torch.float32,
    ).to(device)
    pw_val = torch.tensor(
        [(y_tr[:, 1] == 0).sum() / max((y_tr[:, 1] == 1).sum(), 1)],
        dtype=torch.float32,
    ).to(device)

    model = DeepGAT(
        N_CH, N_FEATS, BACKBONE_DIMS, DENSE_SIZE,
        NUM_HEADS, ATTN_DROPOUT, HEAD_DROPOUT,
    ).to(device)

    criterion = LabelSmoothingBCE(pw_aro, pw_val, smoothing=LABEL_SMOOTHING)
    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-3)

    def lr_lambda(epoch):
        if epoch < WARMUP_EPOCHS:
            return (epoch + 1) / WARMUP_EPOCHS
        progress = (epoch - WARMUP_EPOCHS) / max(EPOCHS - WARMUP_EPOCHS, 1)
        return 0.5 * (1.0 + math.cos(math.pi * progress))

    scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)

    # Training set gets noise augmentation; validation/test never does
    train_dl = DataLoader(
        EEGDataset(x_tr, y_tr, noise_std=NOISE_STD),
        batch_size=BATCH_SIZE, shuffle=True, drop_last=True,
    )
    test_dl = DataLoader(
        EEGDataset(x_te, y_te, noise_std=0.0),
        batch_size=BATCH_SIZE, shuffle=False,
    )

    best_fold_f1     = 0.0
    best_fold_state  = None
    best_t_at_best   = 0.0
    best_v_at_best   = 0.0
    best_val_loss    = float("inf")
    patience_ctr     = 0

    for epoch in range(1, EPOCHS + 1):

        # ── Train ──────────────────────────────────────────────────────────
        model.train()
        t_loss = 0.0
        for Xb, Yb in train_dl:
            Xb, Yb = Xb.to(device), Yb.to(device)
            optimizer.zero_grad()
            loss = criterion(model(Xb), Yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            t_loss += loss.item()
        avg_t = t_loss / len(train_dl)

        # ── Validate ───────────────────────────────────────────────────────
        model.eval()
        v_loss        = 0.0
        all_p, all_tv = [], []
        with torch.no_grad():
            for Xb, Yb in test_dl:
                Xb, Yb  = Xb.to(device), Yb.to(device)
                logits   = model(Xb)
                v_loss  += criterion(logits, Yb).item()
                preds    = (torch.sigmoid(logits) > 0.5).cpu().numpy()
                all_p.append(preds)
                all_tv.append(Yb.cpu().numpy())
        avg_v = v_loss / len(test_dl)

        scheduler.step()

        P        = np.vstack(all_p)
        T        = np.vstack(all_tv)
        f1_aro   = f1_score(T[:, 0], P[:, 0], zero_division=0)
        f1_val_s = f1_score(T[:, 1], P[:, 1], zero_division=0)
        f1_macro = (f1_aro + f1_val_s) / 2

        if f1_macro > best_fold_f1:
            best_fold_f1   = f1_macro
            best_fold_state = {k: v.cpu() for k, v in model.state_dict().items()}
            best_t_at_best = avg_t
            best_v_at_best = avg_v

        # Early stopping on val loss
        if avg_v < best_val_loss:
            best_val_loss = avg_v
            patience_ctr  = 0
        else:
            patience_ctr += 1
            if patience_ctr >= PATIENCE:
                print(f"  Early stop at epoch {epoch}")
                break

        if epoch % 10 == 0 or epoch == EPOCHS:
            ratio = avg_v / avg_t if avg_t > 0 else 999
            print(
                f"  Ep {epoch:03d} | "
                f"Train: {avg_t:.4f} | Val: {avg_v:.4f} | "
                f"Gap: {ratio:.1f}x | "
                f"F1 Aro: {f1_aro:.3f} | F1 Val: {f1_val_s:.3f} | "
                f"F1 Macro: {f1_macro:.3f}"
            )

    fold_results_macro.append(best_fold_f1)
    fold_train_loss_best.append(best_t_at_best)
    fold_val_loss_best.append(best_v_at_best)
    ratio_at_best = best_v_at_best / best_t_at_best if best_t_at_best > 0 else 999
    print(f"  --> Best F1: {best_fold_f1:.4f} | "
          f"Loss gap at best epoch: {ratio_at_best:.1f}x")

    if best_fold_f1 > best_overall_f1:
        best_overall_f1   = best_fold_f1
        best_overall_state = best_fold_state
        best_fold_for_xai  = fold
        x_te_xai           = x_te
        y_te_xai           = y_te
        x_tr_xai           = x_tr
        y_tr_xai           = y_tr

print(f"\n{'='*55}")
print(f"[!] Training complete.")
print(f"[!] Per-fold F1: {[round(v, 4) for v in fold_results_macro]}")
print(f"[!] Mean ± Std : {np.mean(fold_results_macro):.4f} ± {np.std(fold_results_macro):.4f}")
print(f"[!] Best fold  : {best_fold_for_xai} (F1={best_overall_f1:.4f})")

# Overfitting diagnosis across all folds
mean_ratio = np.mean([
    v / t if t > 0 else 999
    for v, t in zip(fold_val_loss_best, fold_train_loss_best)
])
print(f"\n[!] Mean train/val loss ratio at best epoch: {mean_ratio:.1f}x")
if mean_ratio > 5:
    print("    ⚠ Ratio > 5x — model still memorising training windows.")
    print("    Try: increase NOISE_STD to 0.2-0.3, HEAD_DROPOUT to 0.7.")
elif mean_ratio > 2:
    print("    ~ Moderate gap — acceptable for small single-subject EEG datasets.")
else:
    print("    ✓ Loss gap looks healthy.")

torch.save(best_overall_state, os.path.join(OUT_WORK, "best_model.pth"))

# ─────────────────────────────────────────────────────────────────────────────
# 7. FINAL EVALUATION
# ─────────────────────────────────────────────────────────────────────────────
print(f"\nFinal evaluation on best fold ({best_fold_for_xai}) held-out set...")
model.load_state_dict(best_overall_state)
model.to(device)
model.eval()

final_dl = DataLoader(
    EEGDataset(x_te_xai, y_te_xai, noise_std=0.0),
    batch_size=BATCH_SIZE, shuffle=False,
)

all_p, all_t = [], []
with torch.no_grad():
    for Xb, Yb in final_dl:
        logits = model(Xb.to(device))
        preds  = (torch.sigmoid(logits) > 0.5).cpu().numpy()
        all_p.append(preds)
        all_t.append(Yb.numpy())
all_p = np.vstack(all_p)
all_t = np.vstack(all_t)

for i, task in enumerate(["Arousal", "Valence"]):
    print(f"\n── {task} ──")
    print(classification_report(
        all_t[:, i], all_p[:, i], target_names=["Low", "High"]
    ))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for i, task in enumerate(["Arousal", "Valence"]):
    cm = confusion_matrix(all_t[:, i], all_p[:, i])
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues",
        xticklabels=["Low", "High"], yticklabels=["Low", "High"],
        ax=axes[i],
    )
    axes[i].set_title(f"Confusion Matrix — {task}")
    axes[i].set_ylabel("True")
    axes[i].set_xlabel("Predicted")
plt.tight_layout()
plt.savefig(os.path.join(OUT_WORK, "confusion_matrices.png"), dpi=150)
plt.close()

# Fold-level summary plots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

folds = list(range(1, N_SPLITS + 1))
bars  = ax1.bar(folds, fold_results_macro, color="#3498DB", edgecolor="black", alpha=0.85)
ax1.axhline(np.mean(fold_results_macro), color="#C0392B", linewidth=1.5,
            linestyle="--", label=f"Mean={np.mean(fold_results_macro):.3f}")
for bar, v in zip(bars, fold_results_macro):
    ax1.text(bar.get_x() + bar.get_width() / 2, v + 0.005,
             f"{v:.3f}", ha="center", va="bottom", fontsize=8)
ax1.set_title(f"{N_SPLITS}-Fold Macro F1")
ax1.set_xlabel("Fold")
ax1.set_ylabel("Best Macro F1")
ax1.set_ylim(0, 1)
ax1.set_xticks(folds)
ax1.legend()

ax2.plot(folds, fold_train_loss_best, "o-", label="Train loss", color="#C0392B")
ax2.plot(folds, fold_val_loss_best,   "s-", label="Val loss",   color="#2980B9")
ax2.set_title("Train vs Val Loss at Best-F1 Epoch")
ax2.set_xlabel("Fold")
ax2.set_ylabel("Loss")
ax2.set_xticks(folds)
ax2.legend()

plt.tight_layout()
plt.savefig(os.path.join(OUT_WORK, "fold_summary.png"), dpi=150)
plt.close()

# ─────────────────────────────────────────────────────────────────────────────
# 8. XAI — GAT ATTENTION WEIGHTS
# ─────────────────────────────────────────────────────────────────────────────
print("\n[XAI] Extracting GAT attention weights...")

n_xai = min(300, len(x_te_xai))
x_xai = torch.tensor(x_te_xai[:n_xai]).float().to(device)

with torch.no_grad():
    _, bb_attns = model(x_xai, return_attn=True)


def mean_attn(aw_list):
    return [aw.mean(dim=(0, 1)).cpu().numpy() for aw in aw_list]


bb_maps  = mean_attn(bb_attns)
n_layers = len(bb_maps)

n_cols = min(n_layers, 3)
n_rows = math.ceil(n_layers / n_cols)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(7 * n_cols, 6 * n_rows))
axes = np.array(axes).flatten()

for li, mat in enumerate(bb_maps):
    sns.heatmap(
        mat, xticklabels=CHANNEL_NAMES, yticklabels=CHANNEL_NAMES,
        cmap="YlOrRd", vmin=0, annot=False, ax=axes[li],
        cbar=True, linewidths=0.2,
    )
    axes[li].set_title(
        f"Backbone Layer {li+1}\n(α, avg {n_xai} samples)", fontsize=11
    )
for ax in axes[n_layers:]:
    ax.axis("off")

plt.suptitle("Native GAT Attention", fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(OUT_WORK, "xai_backbone_layers.png"), dpi=150)
plt.close()

ch_imp_by_layer = np.array([mat.sum(axis=0) for mat in bb_maps])
fig, ax = plt.subplots(figsize=(14, 6))
cmap = plt.cm.plasma
for li in range(n_layers):
    ax.plot(CHANNEL_NAMES, ch_imp_by_layer[li], marker="o",
            label=f"Layer {li+1}", color=cmap(li / max(n_layers - 1, 1)),
            linewidth=2)
ax.set_title("Channel Attention Importance by Layer", fontsize=13)
ax.set_xlabel("EEG Channel")
ax.set_ylabel("Cumulative Attention Received")
ax.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(os.path.join(OUT_WORK, "xai_channel_evolution.png"), dpi=150)
plt.close()

final_mat = bb_maps[-1]
ch_imp    = final_mat.sum(axis=0)
order     = np.argsort(ch_imp)[::-1]
colors    = plt.cm.plasma(ch_imp[order] / ch_imp.max())
fig, ax   = plt.subplots(figsize=(13, 5))
ax.bar([CHANNEL_NAMES[i] for i in order], ch_imp[order],
       color=colors, edgecolor="black", lw=0.6)
ax.set_title("Per-Channel Attention Importance (Final GAT Layer)", fontsize=13)
ax.set_xlabel("EEG Channel")
ax.set_ylabel("Cumulative Attention (α column-sum)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(os.path.join(OUT_WORK, "xai_channel_importance_bar.png"), dpi=150)
plt.close()
print("  Attention XAI plots saved.")

# ─────────────────────────────────────────────────────────────────────────────
# 9. XAI — SHAP
# ─────────────────────────────────────────────────────────────────────────────
print("[XAI] Running SHAP...")

if torch.cuda.is_available():
    torch.backends.cudnn.enabled = False


class TaskWrapper(nn.Module):
    def __init__(self, base, task_idx):
        super().__init__()
        self.base = base
        self.t    = task_idx

    def forward(self, x):
        return self.base(x)[:, self.t].unsqueeze(1)


n_bg   = min(200, len(x_tr_xai))
n_test = min(150, len(x_te_xai))
bg     = torch.tensor(x_tr_xai[:n_bg]).float().to(device)
test   = torch.tensor(x_te_xai[:n_test]).float().to(device)

shap_data = {}
for t_idx, t_name in enumerate(["Arousal", "Valence"]):
    wrapper   = TaskWrapper(model, t_idx).to(device)
    explainer = shap.GradientExplainer(wrapper, bg)
    sv        = explainer.shap_values(test)
    sv        = sv[0] if isinstance(sv, list) else sv
    while sv.ndim > 3:
        sv = sv.squeeze(-1)

    mean_abs = np.abs(sv).mean(axis=0)
    shap_data[t_name] = {
        "heatmap":    mean_abs,
        "ch_total":   mean_abs.sum(axis=1),
        "feat_total": mean_abs.sum(axis=0),
    }

    fig, ax = plt.subplots(figsize=(14, 8))
    sns.heatmap(
        mean_abs, xticklabels=FEAT_LABELS, yticklabels=CHANNEL_NAMES,
        cmap="plasma", annot=True, fmt=".4f", linewidths=0.3, ax=ax,
    )
    ax.set_title(
        f"SHAP Feature Importance — {t_name}\n"
        f"Left 5 = Band Power  |  Right 5 = Differential Entropy",
        fontsize=13,
    )
    ax.set_xlabel("Feature")
    ax.set_ylabel("EEG Channel")
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_WORK, f"shap_heatmap_{t_name.lower()}.png"), dpi=150)
    plt.close()

x_pos = np.arange(N_CH)
fig, ax = plt.subplots(figsize=(14, 6))
ax.bar(x_pos - 0.2, shap_data["Arousal"]["ch_total"], 0.38,
       label="Arousal", color="#C0392B", edgecolor="black")
ax.bar(x_pos + 0.2, shap_data["Valence"]["ch_total"], 0.38,
       label="Valence", color="#2980B9", edgecolor="black")
ax.set_xticks(x_pos)
ax.set_xticklabels(CHANNEL_NAMES, rotation=45)
ax.set_title("SHAP Channel Importance: Arousal vs Valence", fontsize=13)
ax.set_ylabel("Total |SHAP| per channel")
ax.legend(fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(OUT_WORK, "shap_channel_aro_vs_val.png"), dpi=150)
plt.close()

bp_mean = np.mean(
    [shap_data[t]["feat_total"][:N_BANDS] for t in ["Arousal", "Valence"]], axis=0
)
de_mean = np.mean(
    [shap_data[t]["feat_total"][N_BANDS:] for t in ["Arousal", "Valence"]], axis=0
)
fig, ax = plt.subplots(figsize=(10, 5))
xb = np.arange(N_BANDS)
ax.bar(xb - 0.2, bp_mean, 0.38, label="Band Power",
       color="#27AE60", edgecolor="black")
ax.bar(xb + 0.2, de_mean, 0.38, label="Diff. Entropy",
       color="#8E44AD", edgecolor="black")
ax.set_xticks(xb)
ax.set_xticklabels(BAND_LABELS)
ax.set_title("Feature Type SHAP: Band Power vs DE", fontsize=13)
ax.set_ylabel("Mean |SHAP|")
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUT_WORK, "shap_bp_vs_de.png"), dpi=150)
plt.close()

if torch.cuda.is_available():
    torch.backends.cudnn.enabled = True
print("  All SHAP plots saved.")

# ─────────────────────────────────────────────────────────────────────────────
# SUMMARY
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 65)
print(f"DONE — outputs in: {OUT_WORK}")
print("=" * 65)
print(f"Mean Macro F1 : {np.mean(fold_results_macro):.4f} ± {np.std(fold_results_macro):.4f}")
print(f"Best fold F1  : {best_overall_f1:.4f}  (fold {best_fold_for_xai})")
print(f"\nKey config:")
print(f"  STEP={STEP} (0% overlap) | ARTIFACT_THRESHOLD={ARTIFACT_THRESHOLD} µV")
print(f"  Backbone={BACKBONE_DIMS}, Dense={DENSE_SIZE}, Heads={NUM_HEADS}")
print(f"  noise_std={NOISE_STD}, label_smoothing={LABEL_SMOOTHING}")
print(f"  head_dropout={HEAD_DROPOUT}, weight_decay=1e-3")
print(f"  batch={BATCH_SIZE}, early_stopping patience={PATIENCE}")

Device : cuda
Extracting features into data_extracted_v2 (fresh cache)...


  happy:  36%|████████████████████████████████                                                         | 9/25 [00:01<00:01,  8.64it/s]

  ⚠ No valid windows in clean_trial_17_Uptown_Funk.csv — skipped.


  sad:   8%|███████▎                                                                                   | 2/25 [00:00<00:01, 14.62it/s]

  ⚠ No valid windows in clean_trial_10_Magic.csv — skipped.
  ⚠ No valid windows in clean_trial_12_Poate.csv — skipped.


...

  ⚠ No valid windows in clean_trial_16_Fade_Into_You.csv — skipped.


  sad:  88%|███████████████████████████████████████████████████████████████████████████████▏          | 22/25 [00:01<00:00, 10.41it/s]

  ⚠ No valid windows in clean_trial_7_Nude.csv — skipped.
  ⚠ No valid windows in clean_trial_8_And_I_Love_Her.csv — skipped.


  stressed: 100%|█████████████████████████████████████████████████████████████████████████████████████| 25/25 [00:03<00:00,  6.90it/s]



Loading windows...

───────────────────────────────────────────────────────
  Unique files       : 119
  Total windows      : 4,497
  Windows/file       : 37.8  (expected ~58)
  STEP check         : OK
───────────────────────────────────────────────────────

[!] Starting 10-Fold GroupKFold (file-level splits)
[!] noise_std=0.15 | label_smoothing=0.1 | patience=15 | batch=64

 Fold 1/10 | train=4,048 | test=449
  Ep 010 | Train: 0.3988 | Val: 0.4388 | Gap: 1.1x | F1 Aro: 0.906 | F1 Val: 0.788 | F1 Macro: 0.847
  Ep 020 | Train: 0.3236 | Val: 0.3190 | Gap: 1.0x | F1 Aro: 0.945 | F1 Val: 0.860 | F1 Macro: 0.903
  Ep 030 | Train: 0.2924 | Val: 0.3160 | Gap: 1.1x | F1 Aro: 0.940 | F1 Val: 0.901 | F1 Macro: 0.921
  Ep 040 | Train: 0.2777 | Val: 0.3071 | Gap: 1.1x | F1 Aro: 0.956 | F1 Val: 0.905 | F1 Macro: 0.930
  Early stop at epoch 50
  --> Best F1: 0.9351 | Loss gap at best epoch: 1.1x
 Fold 2/10 | train=4,048 | test=449
  Ep 010 | Train: 0.3685 | Val: 0.3439 | Gap: 0.9x | F1 Aro: 0.687 